<a href="https://colab.research.google.com/github/chandupradeep18/Building-with-the-Claude-API/blob/main/prompt_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 14.0 MB/s eta 0:00:00


In [28]:
from anthropic import Anthropic
from google.colab import userdata
import json
from datetime import  datetime
from statistics import mean
import re
import ast

In [17]:
client=Anthropic(api_key=userdata.get('Claude'))
model='claude-haiku-4-5-20251001'
max_tokens=1000

In [7]:
def add_user_message(messages,text):
  messages.append({"role":"user","content":text})
def add_assistant_message(messages,text):
  messages.append({"role":"assistant","content":text})
def chat(message,system=None,temperature=1.0,stop_sequences=[]):
  param={
      'model':model,
      'max_tokens':max_tokens,
      'temperature':temperature,
      'messages':message
  }
  if system:
    param['system']=system
  if stop_sequences:
    param["stop_sequences"] = stop_sequences
  results=client.messages.create(**param)
  return results.content[0].text

In [21]:
def generate_dataset():
  prompt = """
  Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

  Example output:
  ```json
  [
    {
       "task": "Description of task",
       "format":""
    },
    ...additional
  ]
  ```

  * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
  * Focus on tasks that do not require writing much code

  Please generate 3 objects."""
  messages = []
  add_user_message(messages,prompt)
  add_assistant_message(messages, "```json")
  text = chat(messages, stop_sequences=["```"])
  with open('dataset.json','w') as new_json:
    json.dump(json.loads(text),new_json,indent=2)

In [31]:
generate_dataset()

In [36]:
def run_prompt(test_case):
  prompt = f"""
  Please solve the following task in max token of 500 only:
  {test_case['task']}

  * Respond only with Python, JSON, or a plain Regex
  * Do not add any comments or commentary or explanation
  """
  messages = []
  add_user_message(messages,prompt)
  add_assistant_message(messages, "```code")
  text = chat(messages,stop_sequences=['```'])
  return text

In [32]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [38]:
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0
def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [37]:
def run_test_cases(test_case):
  output=run_prompt(test_case)
  model_grade = grade_by_model(test_case, output)
  score = model_grade["score"]
  reasoning = model_grade["reasoning"]
  syntax_score=grade_syntax(output,test_case)
  return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
        "syntax_score": syntax_score
    }

In [39]:
def run_eval(dataset):
  results=[]
  for testcase in dataset:
    results.append(run_test_cases(testcase))
  score=mean([result['score'] for result in results])
  return results,score

In [40]:
with open('dataset.json','r') as file:
  dataset=json.load(file)
results,score=run_eval(dataset)
with open('results.json','w') as file:
  json.dump(results,file,indent=2)

In [41]:

print(score)
results

6


[{'output': '\nimport re\n\ndef extract_region_from_s3_arn(arn):\n    """\n    Extracts AWS region from an S3 bucket ARN.\n    S3 bucket ARNs don\'t contain region info (always global),\n    but this function handles the general case.\n    """\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    parts = arn.split(\':\')\n    \n    if len(parts) < 6:\n        return None\n    \n    if parts[2] != \'s3\':\n        return None\n    \n    region = parts[3]\n    \n    return region if region else None\n\n\ndef extract_region_from_s3_arn_regex(arn):\n    """\n    Extracts AWS region from an S3 bucket ARN using regex.\n    """\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    match = re.match(r\'arn:aws:s3:([^:]*):([^:]*):([^:]*)\', arn)\n    \n    if match:\n        region = match.group(1)\n        return region if region else None\n    \n    return None\n\n\nif __name__ == "__main__":\n    test_arns = [\n        \'arn:aws:s3:::my-bucket\',\

In [ ]:
print('git commit -m "feat: Add evaluation framework for AI-generated AWS code"')